# Intelligent User Question Answering System
## Using NLP and Transformer-Based Machine Learning

**Dataset:** Stanford Question Answering Dataset (SQuAD) 1.1  
**Model:** distilbert-base-uncased-distilled-squad  
**Submitted to:** WHEELTRIX HR Department  

---

### Notebook Outline
1. Environment Setup
2. Dataset Loading and Parsing
3. Data Quality Checks
4. Exploratory Data Analysis (EDA)
5. Text Preprocessing
6. Tokenisation
7. Model Loading
8. Prediction Demo
9. Evaluation (EM + F1)
10. Results Summary

---
## 1. Environment Setup

In [ ]:
# Install required packages (run once if not already installed)
# !pip install transformers datasets torch pandas matplotlib seaborn wordcloud tqdm

import os
import sys
import json
import re
import string
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
from tqdm.notebook import tqdm

import torch
from transformers import DistilBertTokenizerFast, DistilBertForQuestionAnswering

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
sns.set_theme(style='whitegrid', palette='Blues_d')

# Add project src/ to path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root  : {PROJECT_ROOT}')
print(f'Figures dir   : {FIGURES_DIR}')
print(f'PyTorch       : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

---
## 2. Dataset Loading and Parsing

SQuAD 1.1 is distributed as two JSON files:
- `train-v1.1.json` — 87,599 QA pairs
- `dev-v1.1.json`   — 10,570 QA pairs

The JSON has a **three-level hierarchy**:  
`articles → paragraphs → question-answer pairs`

In [ ]:
from data_preprocessing import (
    download_squad,
    parse_squad_json,
    normalize_dataframe,
    add_length_columns,
    check_data_quality,
    RAW_DIR,
    PROCESSED_DIR,
)

# Download if not present
train_path = download_squad('train')
dev_path   = download_squad('dev')

print(f'Train JSON : {train_path}  ({train_path.stat().st_size / 1e6:.1f} MB)')
print(f'Dev   JSON : {dev_path}    ({dev_path.stat().st_size / 1e6:.1f} MB)')

In [ ]:
# Parse JSON → flat DataFrames
train_df = parse_squad_json(train_path, 'train')
dev_df   = parse_squad_json(dev_path,   'dev')

print(f'\nTrain shape: {train_df.shape}')
print(f'Dev   shape: {dev_df.shape}')
train_df.head(3)

In [ ]:
# Column names and dtypes
print(train_df.dtypes)

---
## 3. Data Quality Checks

In [ ]:
check_data_quality(train_df, 'train')
check_data_quality(dev_df,   'dev')

In [ ]:
# Missing values per column
print('=== Train missing values ===')
print(train_df.isnull().sum())

print('\n=== Dev missing values ===')
print(dev_df.isnull().sum())

In [ ]:
# Duplicate question IDs
print('Train duplicate QIDs:', train_df.duplicated(subset=['question_id']).sum())
print('Dev   duplicate QIDs:', dev_df.duplicated(subset=['question_id']).sum())

# Unique articles
print('\nTrain unique articles:', train_df['title'].nunique())
print('Dev   unique articles:', dev_df['title'].nunique())

---
## 4. Exploratory Data Analysis (EDA)

All charts are saved to `outputs/figures/`.

In [ ]:
# Add length columns
train_df = add_length_columns(train_df)
dev_df   = add_length_columns(dev_df)

combined_df = pd.concat([train_df, dev_df], ignore_index=True)

# Summary statistics
stats = combined_df[['question_word_count', 'context_word_count', 'answer_word_count']].describe().round(2)
print(stats)

In [ ]:
# ── Figure 1: Dataset Split Distribution ─────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
split_counts = combined_df['split'].value_counts()
bars = ax.bar(split_counts.index, split_counts.values,
              color=['#1565c0', '#42a5f5'], edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, split_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_title('Figure 1: SQuAD 1.1 Dataset Split Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Split', fontsize=11)
ax.set_ylabel('Number of QA Pairs', fontsize=11)
ax.set_ylim(0, split_counts.max() * 1.15)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nObservation: The training split contains ~87,599 QA pairs and the dev split ~10,570.')
print('The train:dev ratio is approximately 8.3:1, which is standard for benchmarking.')

In [ ]:
# ── Figure 2: Question Length Distribution ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_df['question_word_count'], bins=30, color='#1565c0',
             edgecolor='white', alpha=0.85)
axes[0].axvline(train_df['question_word_count'].mean(), color='#e53935',
                linestyle='--', linewidth=1.8, label=f"Mean = {train_df['question_word_count'].mean():.1f}")
axes[0].set_title('Train: Question Length (Words)', fontweight='bold')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(dev_df['question_word_count'], bins=30, color='#42a5f5',
             edgecolor='white', alpha=0.85)
axes[1].axvline(dev_df['question_word_count'].mean(), color='#e53935',
                linestyle='--', linewidth=1.8, label=f"Mean = {dev_df['question_word_count'].mean():.1f}")
axes[1].set_title('Dev: Question Length (Words)', fontweight='bold')
axes[1].set_xlabel('Number of Words')
axes[1].legend()

fig.suptitle('Figure 2: Question Word Count Distribution – SQuAD 1.1',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_question_length.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observation: Most questions are 5–15 words long. The distribution is right-skewed,')
print('indicating occasional longer questions but a consistent short-to-medium length pattern.')

In [ ]:
# ── Figure 3: Context Length Distribution ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_df['context_word_count'], bins=50, color='#283593',
        edgecolor='white', alpha=0.85, label='Train')
ax.hist(dev_df['context_word_count'], bins=50, color='#64b5f6',
        edgecolor='white', alpha=0.65, label='Dev')
ax.axvline(train_df['context_word_count'].mean(), color='#e53935',
           linestyle='--', linewidth=1.8,
           label=f"Train mean = {train_df['context_word_count'].mean():.0f} words")
ax.set_title('Figure 3: Context (Passage) Word Count Distribution – SQuAD 1.1',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Words in Context')
ax.set_ylabel('Count')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_context_length.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Train context avg: {train_df["context_word_count"].mean():.1f} words')
print(f'Dev   context avg: {dev_df["context_word_count"].mean():.1f} words')
print('Observation: Contexts range from ~50 to ~700 words, with most between 100–250 words.')

In [ ]:
# ── Figure 4: Answer Length Distribution ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(train_df['answer_word_count'], bins=30, range=(0, 30),
        color='#1b5e20', edgecolor='white', alpha=0.85)
ax.axvline(train_df['answer_word_count'].mean(), color='#e53935',
           linestyle='--', linewidth=2,
           label=f"Mean = {train_df['answer_word_count'].mean():.1f} words")
ax.set_title('Figure 4: Answer Word Count Distribution – SQuAD 1.1 Train',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Words in Answer')
ax.set_ylabel('Count')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_answer_length.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Answer avg: {train_df["answer_word_count"].mean():.1f} words')
print('Observation: Answers are predominantly 1–5 words (single entity or short phrase),')
print('confirming SQuAD is an extractive span-based dataset.')

In [ ]:
# ── Figure 5: Top 20 Frequent Words in Questions ─────────────────────────
STOPWORDS = {'what', 'which', 'who', 'when', 'where', 'how', 'why', 'did',
             'do', 'does', 'is', 'are', 'was', 'were', 'the', 'a', 'an',
             'in', 'of', 'to', 'for', 'on', 'at', 'by', 'from', 'with', ''}

q_words = (
    train_df['question']
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.split()
    .explode()
)
q_freq = (
    q_words[~q_words.isin(STOPWORDS)]
    .value_counts()
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 5))
q_freq.sort_values().plot(kind='barh', ax=ax, color='#1565c0', edgecolor='white')
ax.set_title('Figure 5: Top 20 Content Words in Questions (Train Split)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Frequency')
ax.set_ylabel('Word')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_top_question_words.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observation: Common question words include many, name, year, type,')
print('revealing that SQuAD questions ask for counts, names, dates, and classifications.')

In [ ]:
# ── Figure 6: Question Word Cloud ─────────────────────────────────────────
q_text = ' '.join(
    train_df['question']
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
)
wc = WordCloud(width=900, height=400, background_color='white',
               colormap='Blues', max_words=150,
               stopwords=STOPWORDS).generate(q_text)

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Figure 6: Word Cloud – SQuAD 1.1 Questions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig6_question_wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observation: The word cloud confirms the dominance of factual query words.')

In [ ]:
# ── Figure 7: Answer Length vs Question Length scatter ────────────────────
sample = train_df.sample(n=3000, random_state=42)
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(sample['question_word_count'], sample['answer_word_count'],
                alpha=0.3, s=15, c='#1565c0')
ax.set_title('Figure 7: Answer Length vs Question Length (3 000-sample)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Question Word Count')
ax.set_ylabel('Answer Word Count')
ax.set_xlim(0, 40)
ax.set_ylim(0, 30)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig7_answer_vs_question_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observation: No strong linear correlation – long questions can have very short answers.')

In [ ]:
# ── Figure 8: Top 15 Articles ─────────────────────────────────────────────
top_articles = train_df['title'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 5))
top_articles.sort_values().plot(kind='barh', ax=ax, color='#283593', edgecolor='white')
ax.set_title('Figure 8: Top 15 Wikipedia Articles by QA Pair Count (Train)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of QA Pairs')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig8_top_articles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary statistics table ──────────────────────────────────────────────
summary = pd.DataFrame({
    'Metric': [
        'Total QA pairs (Train)', 'Total QA pairs (Dev)', 'Total QA pairs',
        'Unique contexts (Train)', 'Unique contexts (Dev)',
        'Unique articles (Train)', 'Unique articles (Dev)',
        'Avg question length (words, Train)', 'Avg context length (words, Train)',
        'Avg answer length (words, Train)',
        'Max question length (words)', 'Max context length (words)',
    ],
    'Value': [
        len(train_df), len(dev_df), len(train_df) + len(dev_df),
        train_df['context'].nunique(), dev_df['context'].nunique(),
        train_df['title'].nunique(), dev_df['title'].nunique(),
        f"{train_df['question_word_count'].mean():.1f}",
        f"{train_df['context_word_count'].mean():.1f}",
        f"{train_df['answer_word_count'].mean():.1f}",
        train_df['question_word_count'].max(),
        train_df['context_word_count'].max(),
    ]
})
print(summary.to_string(index=False))

---
## 5. Text Preprocessing

In [ ]:
from data_preprocessing import normalize_dataframe

# Normalise (collapse whitespace, strip)
train_df = normalize_dataframe(train_df)
dev_df   = normalize_dataframe(dev_df)

# Show before / after for one example
sample_idx = 42
raw_q = train_df.iloc[sample_idx]['question']
print(f'Normalised question : {raw_q}')
print(f'Answer              : {train_df.iloc[sample_idx]["answer_text"]}')

---
## 6. Tokenisation

DistilBERT uses WordPiece tokenisation.  
The tokeniser encodes the question and context as:  
`[CLS] question tokens [SEP] context tokens [SEP]`

We use `return_offsets_mapping=True` to map predicted token positions  
back to character positions in the original context.

In [ ]:
from data_preprocessing import get_tokenizer, encode_sample

tokenizer = get_tokenizer()

# Show tokenisation for one example
row = train_df.iloc[0]
print(f'Question : {row["question"]}')
print(f'Answer   : {row["answer_text"]}')
print(f'Answer start (char): {row["answer_start"]}')

encoded = encode_sample(
    tokenizer=tokenizer,
    question=row['question'],
    context=row['context'],
    answer_text=row['answer_text'],
    answer_start=int(row['answer_start']),
)

print(f'\ninput_ids length    : {len(encoded["input_ids"])}')
print(f'start_position      : {encoded["start_positions"]}')
print(f'end_position        : {encoded["end_positions"]}')

# Decode the predicted span for visual verification
span_tokens = encoded['input_ids'][encoded['start_positions'] : encoded['end_positions'] + 1]
decoded_span = tokenizer.decode(span_tokens)
print(f'Token span decoded  : {decoded_span}')

---
## 7. Model Loading

In [ ]:
from model import QAModel, get_device

device = get_device()

qa_model = QAModel()
qa_model.load_pretrained()

# Print model summary
total_params = sum(p.numel() for p in qa_model.model.parameters())
trainable_params = sum(p.numel() for p in qa_model.model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

---
## 8. Prediction Demo

In [ ]:
# Demo predictions on curated examples
demo_pairs = [
    {
        'question': 'What is SQuAD?',
        'context': (
            'The Stanford Question Answering Dataset (SQuAD) is a reading comprehension '
            'dataset consisting of questions posed by crowdworkers on a set of Wikipedia '
            'articles, where the answer to every question is a segment of text from the '
            'corresponding reading passage. SQuAD 1.1 contains 100,000+ question-answer '
            'pairs on 500+ articles.'
        )
    },
    {
        'question': 'Who introduced the Transformer architecture?',
        'context': (
            'The Transformer architecture was introduced by Vaswani et al. in the '
            "2017 paper 'Attention Is All You Need'. It relies entirely on a "
            'self-attention mechanism to draw global dependencies between input '
            'and output.'
        )
    },
    {
        'question': 'When was Python first released?',
        'context': (
            'Python is a high-level, general-purpose programming language. '
            'Python was created by Guido van Rossum and first released in 1991.'
        )
    },
]

print(f'{"="*65}')
print(f'{"  DEMO PREDICTIONS":^65}')
print(f'{"="*65}')

for i, pair in enumerate(demo_pairs, 1):
    result = qa_model.predict(pair['question'], pair['context'])
    print(f'\nExample {i}')
    print(f'  Question   : {pair["question"]}')
    print(f'  Answer     : {result["answer"]}')
    print(f'  Confidence : {result["score"]:.4f}')

---
## 9. Evaluation (Exact Match + F1)

We evaluate on a **500-sample subset of the dev split** to keep runtime  
manageable on a laptop CPU (~10–20 minutes).  
Run `python src/evaluate.py` without `--sample` to evaluate the full dev set.

In [ ]:
from evaluate import evaluate_model, save_results, compute_exact, compute_f1

EVAL_SAMPLE = 200   # Increase to 500 or remove cap for more accurate results

print(f'Evaluating on {EVAL_SAMPLE} dev examples …')
results = evaluate_model(qa_model, dev_df, sample_size=EVAL_SAMPLE)

print(f'\n── Evaluation Results ──')
print(f'  Examples    : {results["num_examples"]}')
print(f'  Exact Match : {results["exact_match"]:.2f}%')
print(f'  F1 Score    : {results["f1"]:.2f}%')

In [ ]:
# Save results to outputs/results/
save_results(results, filename_stem='notebook_evaluation')

# Show sample predictions
pred_df = pd.DataFrame(results['predictions'])
pred_df[['question', 'gold_answer', 'predicted_answer', 'exact_match', 'f1_score']].head(10)

---
## 10. Results Summary

In [ ]:
# Results table
results_table = pd.DataFrame({
    'Metric': ['Exact Match (EM)', 'F1 Score'],
    'Score (%)': [results['exact_match'], results['f1']],
    'Description': [
        'Fraction of predictions that exactly match the gold answer',
        'Average token-level overlap between prediction and gold answer',
    ]
})
print('\n── Evaluation Results Table ──')
print(results_table.to_string(index=False))
print(f'\nNote: Results computed on {EVAL_SAMPLE} dev samples.')
print('For full dev evaluation, run: python src/evaluate.py')

In [ ]:
# ── Figure 9: EM vs F1 bar chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
metrics = ['Exact Match', 'F1 Score']
values  = [results['exact_match'], results['f1']]
colors  = ['#1565c0', '#43a047']
bars = ax.bar(metrics, values, color=colors, edgecolor='white', linewidth=1.2, width=0.45)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylim(0, 105)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_title(f'Figure 9: Evaluation Results ({EVAL_SAMPLE} dev examples)',
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig9_evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: The pretrained distilbert-base-uncased-distilled-squad achieves')
print('EM ≈ 79-80%  and  F1 ≈ 86-87% on the FULL dev set.')

---
## Notebook Complete ✅

**Next steps:**
- To fine-tune: `python src/train.py --sample 2000 --epochs 1`
- To evaluate fully: `python src/evaluate.py`
- To run the Streamlit app: `streamlit run app/app.py`